# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields
print('Available record sets and their fields:')
record_set_ids = []
for rs in metadata.record_sets:
    print(f"- RecordSet @id: {rs.id}; name: {rs.name}")
    record_set_ids.append(rs.id)
    print("  Fields:")
    for f in rs.fields:
        print(f"    - Field @id: {f.id}; name: {f.name}; data_type: {f.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, we will extract and preview data from all available record sets

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f'First 5 rows of RecordSet @id: {record_set_id}')
    print(df.head())
    print('\nColumns:', df.columns.tolist())
    print('---')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

For demonstration, select the first record set containing GAD-7, PHQ-9, or PSQ scores (if present) as the primary set for EDA. All fields are referenced by their `@id`.

In [ ]:
# Identify fields related to scores and select a record set for analysis
primary_record_set_id = None
score_field_ids = []  # e.g. GAD-7, PHQ-9, PSQ

for rs in metadata.record_sets:
    score_fields = [f for f in rs.fields if ('gad' in f.id.lower() or 'phq' in f.id.lower() or 'psq' in f.id.lower() or 'score' in f.name.lower())]
    if score_fields:
        primary_record_set_id = rs.id
        score_field_ids = [f.id for f in score_fields]
        print(f"Selected RecordSet @id: {primary_record_set_id}")
        print("Fields containing scores:")
        for f in score_fields:
            print(f"- Field @id: {f.id}; name: {f.name}; data_type: {f.data_type}")
        break

if not primary_record_set_id:
    # Fallback: use first record set available
    primary_record_set_id = record_set_ids[0]
    rs = [r for r in metadata.record_sets if r.id == primary_record_set_id][0]
    score_field_ids = [f.id for f in rs.fields if f.data_type in ['number', 'integer', 'float', 'double']]
    print(f"Defaulting to RecordSet @id: {primary_record_set_id}")
    print('Numeric field ids:', score_field_ids)

df = dataframes[primary_record_set_id]

# If a score field is found, choose the first for demonstration
if score_field_ids:
    numeric_field_id = score_field_ids[0]
else:
    print('No candidate numeric field found for EDA!')
    numeric_field_id = df.select_dtypes(include=['number', 'float', 'int']).columns[0] if len(df) else None

if numeric_field_id and numeric_field_id in df.columns:
    # Drop missing values
    df = df.dropna(subset=[numeric_field_id])
    # Filter for values above a threshold (use threshold for PHQ-9/GAD-7 e.g. >=10 as an example)
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the field
    field_normalized = f"{numeric_field_id}_normalized"
    filtered_df[field_normalized] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, field_normalized]].head())

    # If a grouping/categorical variable exists (e.g. gender field)
    group_field = None
    for f in [f for f in rs.fields if f.data_type == 'string']:
        if 'gender' in f.id.lower() or 'sex' in f.id.lower() or 'group' in f.id.lower() or 'marital' in f.id.lower():
            group_field = f.id
            break
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field}:")
        print(grouped_df.head())
else:
    print(f"No suitable numeric field for EDA in RecordSet {primary_record_set_id}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Simple distribution plot for the selected numeric field
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouped_df exists, visualize group mean
    if 'grouped_df' in locals():
        plt.figure(figsize=(8, 4))
        sns.barplot(x=group_field, y=numeric_field_id, data=grouped_df)
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, and visualize the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the `mlcroissant` library. Fields and record sets were referenced by their Croissant `@id` throughout. Further analytical work can now be conducted on the processed and visualized data sets.